Script para analizar los resultados de transcripción y mejorar las funciones de normalización
para recalcular WER y CER con mejor precisión.

In [2]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import Tuple
import warnings
warnings.filterwarnings('ignore')

# Configuración
OUTPUT_DIR = Path("output/test_results")
RESULTS_FILE = OUTPUT_DIR / "transcription_results.csv"
OUTPUT_IMPROVED_CSV = OUTPUT_DIR / "transcription_results_improved.csv"
OUTPUT_STATS_TXT = OUTPUT_DIR / "transcription_stats_improved.txt"
OUTPUT_PLOT_METRICS = OUTPUT_DIR / "comparison_metrics.png"
OUTPUT_PLOT_BOXPLOT = OUTPUT_DIR / "comparison_boxplot.png"

In [3]:
def normalize_text_improved(text: str) -> str:
    """
    Normalización mejorada del texto para comparación:
    - Convertir a minúsculas
    - Remover puntuación al final de palabras (puntos, comas, etc.)
    - Normalizar espacios múltiples
    - Remover caracteres especiales innecesarios
    """
    if pd.isna(text) or not text:
        return ""
    
    text = str(text)
    
    # Convertir a minúsculas
    text = text.lower()
    
    # Remover puntuación al final de palabras (pero mantener dentro de palabras)
    # Esto quita puntos, comas, punto y coma, dos puntos, etc. al final de palabras
    text = re.sub(r'([a-záéíóúñü])([.,;:!?¡¿]+)(?=\s|$)', r'\1', text)
    
    # Remover puntuación al inicio de palabras (como comillas de apertura)
    text = re.sub(r'(^|\s)([¡¿"\'«»]+)([a-záéíóúñü])', r'\1\3', text)
    
    # Remover puntuación al final de la frase completa
    text = re.sub(r'[.,;:!?¡¿]+$', '', text)
    
    # Normalizar comillas y guiones
    text = text.replace('"', '').replace("'", '').replace('«', '').replace('»', '')
    text = text.replace('—', ' ').replace('–', ' ').replace('-', ' ')
    
    # Remover espacios múltiples y normalizar
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

In [4]:
def calculate_wer(reference: str, hypothesis: str) -> Tuple[float, int, int, int]:
    """
    Calcula Word Error Rate (WER)
    Returns: (WER, substitutions, insertions, deletions)
    """
    if not reference and not hypothesis:
        return 0.0, 0, 0, 0
    
    ref_words = reference.split() if reference else []
    hyp_words = hypothesis.split() if hypothesis else []
    
    # Algoritmo de Levenshtein para palabras
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    
    # Inicializar primera fila y columna
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    
    # Llenar la matriz
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_words[i-1] == hyp_words[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(
                    dp[i-1][j],      # deletion
                    dp[i][j-1],      # insertion
                    dp[i-1][j-1]     # substitution
                )
    
    # Calcular errores
    substitutions = insertions = deletions = 0
    i, j = n, m
    while i > 0 or j > 0:
        if i > 0 and j > 0 and ref_words[i-1] == hyp_words[j-1]:
            i -= 1
            j -= 1
        elif i > 0 and j > 0 and dp[i][j] == dp[i-1][j-1] + 1:
            substitutions += 1
            i -= 1
            j -= 1
        elif j > 0 and dp[i][j] == dp[i][j-1] + 1:
            insertions += 1
            j -= 1
        else:
            deletions += 1
            i -= 1
    
    total_words = len(ref_words)
    wer = (dp[n][m] / total_words * 100) if total_words > 0 else 0.0
    
    return wer, substitutions, insertions, deletions

In [5]:
def calculate_cer(reference: str, hypothesis: str) -> float:
    """
    Calcula Character Error Rate (CER)
    """
    if not reference and not hypothesis:
        return 0.0
    
    ref_chars = list(reference.replace(' ', ''))
    hyp_chars = list(hypothesis.replace(' ', ''))
    
    n, m = len(ref_chars), len(hyp_chars)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    
    for i in range(n + 1):
        dp[i][0] = i
    for j in range(m + 1):
        dp[0][j] = j
    
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if ref_chars[i-1] == hyp_chars[j-1]:
                dp[i][j] = dp[i-1][j-1]
            else:
                dp[i][j] = 1 + min(dp[i-1][j], dp[i][j-1], dp[i-1][j-1])
    
    total_chars = len(ref_chars)
    cer = (dp[n][m] / total_chars * 100) if total_chars > 0 else 0.0
    
    return cer

In [6]:
print("=" * 60)
print("ANÁLISIS DE RESULTADOS DE TRANSCRIPCIÓN")
print("=" * 60)

# 1. Cargar datos
print("\n[1/8] Cargando datos...")
if not RESULTS_FILE.exists():
    print(f"❌ Error: No se encontró el archivo {RESULTS_FILE}")

df = pd.read_csv(RESULTS_FILE)
print(f"✅ Cargados {len(df)} registros")
print(f"   Columnas: {list(df.columns)}")

ANÁLISIS DE RESULTADOS DE TRANSCRIPCIÓN

[1/8] Cargando datos...
✅ Cargados 6118 registros
   Columnas: ['audio_file', 'reference', 'reference_normalized', 'transcribed', 'transcribed_normalized', 'wer', 'cer', 'substitutions', 'insertions', 'deletions', 'time', 'error']


In [7]:
# 2. Análisis exploratorio inicial
print("\n[2/8] Análisis exploratorio inicial...")
muestras_validas = df['wer'].notna().sum()
muestras_fallidas = df['wer'].isna().sum()
print(f"   Muestras válidas: {muestras_validas}")
print(f"   Muestras fallidas: {muestras_fallidas}")
print(f"\n   WER actual:")
print(f"     Media: {df['wer'].mean():.2f}%")
print(f"     Mediana: {df['wer'].median():.2f}%")
print(f"     Desv. Est.: {df['wer'].std():.2f}%")
print(f"\n   CER actual:")
print(f"     Media: {df['cer'].mean():.2f}%")
print(f"     Mediana: {df['cer'].median():.2f}%")
print(f"     Desv. Est.: {df['cer'].std():.2f}%")


[2/8] Análisis exploratorio inicial...
   Muestras válidas: 6040
   Muestras fallidas: 78

   WER actual:
     Media: 17.06%
     Mediana: 9.09%
     Desv. Est.: 25.30%

   CER actual:
     Media: 7.08%
     Mediana: 2.17%
     Desv. Est.: 16.06%


In [8]:
# 3. Aplicar normalización mejorada
print("\n[3/8] Aplicando normalización mejorada...")
df_improved = df.copy()
df_improved['reference_normalized_improved'] = df_improved['reference'].apply(normalize_text_improved)
df_improved['transcribed_normalized_improved'] = df_improved['transcribed'].apply(normalize_text_improved)
print("✅ Normalización completada")

# Mostrar algunos ejemplos
print("\n   Ejemplos de normalización:")
ejemplos = df_improved.head(5)
for idx, row in ejemplos.iterrows():
    print(f"\n   Ejemplo {idx + 1}:")
    print(f"     Original: {row['reference'][:60]}...")
    print(f"     Normalizado: {row['reference_normalized_improved'][:60]}...")


[3/8] Aplicando normalización mejorada...
✅ Normalización completada

   Ejemplos de normalización:

   Ejemplo 1:
     Original: Orarás á él, y él te oirá; Y tú pagarás tus votos....
     Normalizado: orarás á él y él te oirá y tú pagarás tus votos...

   Ejemplo 2:
     Original: Era preferible un rey respaldado por la tradición española q...
     Normalizado: era preferible un rey respaldado por la tradición española q...

   Ejemplo 3:
     Original: Ha sido profesor universitario en Venezuela, Estados Unidos ...
     Normalizado: ha sido profesor universitario en venezuela estados unidos y...

   Ejemplo 4:
     Original: En la Universidad de El Salvador, obtuvo el grado de Licenci...
     Normalizado: en la universidad de el salvador obtuvo el grado de licencia...

   Ejemplo 5:
     Original: Todo el texto está escrito en tinta de oro....
     Normalizado: todo el texto está escrito en tinta de oro...


In [10]:
# 4. Recalcular WER y CER con normalización mejorada
print("\n[4/8] Recalculando WER y CER con normalización mejorada...")

def calculate_metrics_improved(row):
    """Calcula métricas mejoradas para una fila"""
    ref_norm = row['reference_normalized_improved']
    hyp_norm = row['transcribed_normalized_improved']
    
    if pd.isna(row['wer']):
        # Caso fallido, mantener valores None
        return pd.Series({
            'wer_improved': None,
            'cer_improved': None,
            'substitutions_improved': 0,
            'insertions_improved': 0,
            'deletions_improved': 0
        })
    else:
        wer_improved, subs_improved, ins_improved, dels_improved = calculate_wer(ref_norm, hyp_norm)
        cer_improved = calculate_cer(ref_norm, hyp_norm)
        return pd.Series({
            'wer_improved': wer_improved,
            'cer_improved': cer_improved,
            'substitutions_improved': subs_improved,
            'insertions_improved': ins_improved,
            'deletions_improved': dels_improved
        })

# Aplicar función a cada fila
print("   Procesando muestras...")
metrics_improved = df_improved.apply(calculate_metrics_improved, axis=1)

# Añadir columnas al dataframe
df_improved['wer_improved'] = metrics_improved['wer_improved']
df_improved['cer_improved'] = metrics_improved['cer_improved']
df_improved['substitutions_improved'] = metrics_improved['substitutions_improved']
df_improved['insertions_improved'] = metrics_improved['insertions_improved']
df_improved['deletions_improved'] = metrics_improved['deletions_improved']

print("✅ Recálculo completado")


[4/8] Recalculando WER y CER con normalización mejorada...
   Procesando muestras...
✅ Recálculo completado


In [11]:
# 5. Comparación de resultados
print("\n[5/8] Comparando resultados: Antes vs Después...")
df_comparison = df_improved[df_improved['wer'].notna()].copy()

print(f"\n   Total de muestras válidas: {len(df_comparison)}")
print(f"\n   --- WER (Word Error Rate) ---")
print(f"   Original:")
print(f"     Media: {df_comparison['wer'].mean():.2f}%")
print(f"     Mediana: {df_comparison['wer'].median():.2f}%")
print(f"     Desv. Est.: {df_comparison['wer'].std():.2f}%")
print(f"   Mejorado:")
print(f"     Media: {df_comparison['wer_improved'].mean():.2f}%")
print(f"     Mediana: {df_comparison['wer_improved'].median():.2f}%")
print(f"     Desv. Est.: {df_comparison['wer_improved'].std():.2f}%")

print(f"\n   --- CER (Character Error Rate) ---")
print(f"   Original:")
print(f"     Media: {df_comparison['cer'].mean():.2f}%")
print(f"     Mediana: {df_comparison['cer'].median():.2f}%")
print(f"     Desv. Est.: {df_comparison['cer'].std():.2f}%")
print(f"   Mejorado:")
print(f"     Media: {df_comparison['cer_improved'].mean():.2f}%")
print(f"     Mediana: {df_comparison['cer_improved'].median():.2f}%")
print(f"     Desv. Est.: {df_comparison['cer_improved'].std():.2f}%")

# Calcular mejoras
wer_diff = df_comparison['wer'] - df_comparison['wer_improved']
cer_diff = df_comparison['cer'] - df_comparison['cer_improved']

print(f"\n   --- MEJORAS ---")
print(f"   WER mejorado en promedio: {wer_diff.mean():.2f} puntos porcentuales")
print(f"   CER mejorado en promedio: {cer_diff.mean():.2f} puntos porcentuales")
print(f"   Casos donde WER mejoró: {(wer_diff > 0).sum()} ({(wer_diff > 0).sum() / len(df_comparison) * 100:.1f}%)")
print(f"   Casos donde CER mejoró: {(cer_diff > 0).sum()} ({(cer_diff > 0).sum() / len(df_comparison) * 100:.1f}%)")


[5/8] Comparando resultados: Antes vs Después...

   Total de muestras válidas: 6040

   --- WER (Word Error Rate) ---
   Original:
     Media: 17.06%
     Mediana: 9.09%
     Desv. Est.: 25.30%
   Mejorado:
     Media: 12.98%
     Mediana: 0.00%
     Desv. Est.: 23.90%

   --- CER (Character Error Rate) ---
   Original:
     Media: 7.08%
     Mediana: 2.17%
     Desv. Est.: 16.06%
   Mejorado:
     Media: 6.18%
     Mediana: 0.00%
     Desv. Est.: 16.08%

   --- MEJORAS ---
   WER mejorado en promedio: 4.08 puntos porcentuales
   CER mejorado en promedio: 0.91 puntos porcentuales
   Casos donde WER mejoró: 1716 (28.4%)
   Casos donde CER mejoró: 2036 (33.7%)


In [12]:
 # 6. Ejemplos de mejoras significativas
print("\n[6/8] Analizando casos con mejora significativa...")
mejoras_significativas = df_comparison[wer_diff > 5].sort_values('wer_improved', ascending=True).head(5)
print(f"   Encontrados {len(df_comparison[wer_diff > 5])} casos con mejora > 5 puntos")
if len(mejoras_significativas) > 0:
    print("\n   Top 5 mejoras:")
    for idx, row in mejoras_significativas.iterrows():
        print(f"\n   Caso {idx}:")
        print(f"     Reference: {row['reference'][:70]}...")
        print(f"     Transcribed: {row['transcribed'][:70]}...")
        print(f"     WER: {row['wer']:.2f}% -> {row['wer_improved']:.2f}% (mejora: {row['wer'] - row['wer_improved']:.2f}%)")
        print(f"     CER: {row['cer']:.2f}% -> {row['cer_improved']:.2f}% (mejora: {row['cer'] - row['cer_improved']:.2f}%)")
 


[6/8] Analizando casos con mejora significativa...
   Encontrados 1671 casos con mejora > 5 puntos

   Top 5 mejoras:

   Caso 3:
     Reference: En la Universidad de El Salvador, obtuvo el grado de Licenciado en Rel...
     Transcribed: En la Universidad de El Salvador obtuvo el grado de licenciado en Rela...
     WER: 7.14% -> 0.00% (mejora: 7.14%)
     CER: 1.23% -> 0.00% (mejora: 1.23%)

   Caso 3510:
     Reference: Entra en una ciudad, espera un momento, y luego aparece en un tanque....
     Transcribed: Entra en una ciudad, espera un momento y luego aparece en un tanque....
     WER: 7.69% -> 0.00% (mejora: 7.69%)
     CER: 1.75% -> 0.00% (mejora: 1.75%)

   Caso 3498:
     Reference: Finalmente, se descubre que fue Susana la que entonó la señal....
     Transcribed: Finalmente se descubre que fue Susana la que entonó la señal....
     WER: 9.09% -> 0.00% (mejora: 9.09%)
     CER: 1.92% -> 0.00% (mejora: 1.92%)

   Caso 3485:
     Reference: La poesía tiene un fin, que es el de

In [13]:
# 7. Generar visualizaciones
print("\n[7/8] Generando visualizaciones...")
try:
    # Configurar estilo
    plt.style.use('seaborn-v0_8')
    sns.set_palette("husl")
    
    # Figura 1: Histogramas y scatter plots
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    
    # 1. Distribución de WER: Original vs Mejorado
    axes[0, 0].hist(df_comparison['wer'], bins=50, alpha=0.6, label='Original', color='blue')
    axes[0, 0].hist(df_comparison['wer_improved'], bins=50, alpha=0.6, label='Mejorado', color='green')
    axes[0, 0].set_xlabel('WER (%)')
    axes[0, 0].set_ylabel('Frecuencia')
    axes[0, 0].set_title('Distribución de WER: Original vs Mejorado')
    axes[0, 0].legend()
    axes[0, 0].set_xlim(0, 100)
    
    # 2. Distribución de CER: Original vs Mejorado
    axes[0, 1].hist(df_comparison['cer'], bins=50, alpha=0.6, label='Original', color='blue')
    axes[0, 1].hist(df_comparison['cer_improved'], bins=50, alpha=0.6, label='Mejorado', color='green')
    axes[0, 1].set_xlabel('CER (%)')
    axes[0, 1].set_ylabel('Frecuencia')
    axes[0, 1].set_title('Distribución de CER: Original vs Mejorado')
    axes[0, 1].legend()
    axes[0, 1].set_xlim(0, 50)
    
    # 3. Scatter plot: WER Original vs Mejorado
    axes[1, 0].scatter(df_comparison['wer'], df_comparison['wer_improved'], alpha=0.3, s=10)
    axes[1, 0].plot([0, 200], [0, 200], 'r--', label='Sin cambio')
    axes[1, 0].set_xlabel('WER Original (%)')
    axes[1, 0].set_ylabel('WER Mejorado (%)')
    axes[1, 0].set_title('WER: Original vs Mejorado')
    axes[1, 0].legend()
    axes[1, 0].set_xlim(0, 150)
    axes[1, 0].set_ylim(0, 150)
    
    # 4. Scatter plot: CER Original vs Mejorado
    axes[1, 1].scatter(df_comparison['cer'], df_comparison['cer_improved'], alpha=0.3, s=10)
    axes[1, 1].plot([0, 100], [0, 100], 'r--', label='Sin cambio')
    axes[1, 1].set_xlabel('CER Original (%)')
    axes[1, 1].set_ylabel('CER Mejorado (%)')
    axes[1, 1].set_title('CER: Original vs Mejorado')
    axes[1, 1].legend()
    axes[1, 1].set_xlim(0, 80)
    axes[1, 1].set_ylim(0, 80)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_PLOT_METRICS, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Gráfico guardado: {OUTPUT_PLOT_METRICS}")
    
    # Figura 2: Boxplots
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # WER
    wer_data = [df_comparison['wer'], df_comparison['wer_improved']]
    axes[0].boxplot(wer_data, labels=['Original', 'Mejorado'])
    axes[0].set_ylabel('WER (%)')
    axes[0].set_title('Comparación de WER: Boxplot')
    axes[0].set_ylim(0, 100)
    
    # CER
    cer_data = [df_comparison['cer'], df_comparison['cer_improved']]
    axes[1].boxplot(cer_data, labels=['Original', 'Mejorado'])
    axes[1].set_ylabel('CER (%)')
    axes[1].set_title('Comparación de CER: Boxplot')
    axes[1].set_ylim(0, 50)
    
    plt.tight_layout()
    plt.savefig(OUTPUT_PLOT_BOXPLOT, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✅ Gráfico guardado: {OUTPUT_PLOT_BOXPLOT}")
    
except Exception as e:
    print(f"   ⚠️  Error generando visualizaciones: {e}")
    print("   Continuando sin gráficos...")
    


[7/8] Generando visualizaciones...
   ✅ Gráfico guardado: output\test_results\comparison_metrics.png
   ✅ Gráfico guardado: output\test_results\comparison_boxplot.png


In [14]:
# 8. Guardar resultados
print("\n[8/8] Guardando resultados...")

# Guardar CSV con resultados mejorados
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df_improved.to_csv(OUTPUT_IMPROVED_CSV, index=False, encoding='utf-8')
print(f"   ✅ CSV guardado: {OUTPUT_IMPROVED_CSV}")

# Guardar estadísticas mejoradas
with open(OUTPUT_STATS_TXT, 'w', encoding='utf-8') as f:
    f.write("=" * 60 + "\n")
    f.write("ESTADÍSTICAS DE TRANSCRIPCIÓN (MEJORADAS)\n")
    f.write("=" * 60 + "\n\n")
    f.write(f"Total de muestras: {len(df_improved)}\n")
    f.write(f"Muestras válidas: {df_comparison['wer_improved'].notna().sum()}\n")
    f.write(f"Muestras fallidas: {df_comparison['wer_improved'].isna().sum()}\n\n")
    
    valid_improved = df_comparison['wer_improved'].notna()
    if valid_improved.sum() > 0:
        f.write("Word Error Rate (WER) Mejorado:\n")
        f.write(f"  Media: {df_comparison['wer_improved'].mean():.2f}%\n")
        f.write(f"  Mediana: {df_comparison['wer_improved'].median():.2f}%\n")
        f.write(f"  Desviación estándar: {df_comparison['wer_improved'].std():.2f}%\n")
        f.write(f"  Mínimo: {df_comparison['wer_improved'].min():.2f}%\n")
        f.write(f"  Máximo: {df_comparison['wer_improved'].max():.2f}%\n\n")
        
        f.write("Character Error Rate (CER) Mejorado:\n")
        f.write(f"  Media: {df_comparison['cer_improved'].mean():.2f}%\n")
        f.write(f"  Mediana: {df_comparison['cer_improved'].median():.2f}%\n")
        f.write(f"  Desviación estándar: {df_comparison['cer_improved'].std():.2f}%\n")
        f.write(f"  Mínimo: {df_comparison['cer_improved'].min():.2f}%\n")
        f.write(f"  Máximo: {df_comparison['cer_improved'].max():.2f}%\n\n")
    
    f.write("\nComparación con Métricas Originales:\n")
    f.write(f"  WER mejorado en promedio: {wer_diff.mean():.2f} puntos porcentuales\n")
    f.write(f"  CER mejorado en promedio: {cer_diff.mean():.2f} puntos porcentuales\n")
    f.write(f"  Casos donde WER mejoró: {(wer_diff > 0).sum()} ({(wer_diff > 0).sum() / len(df_comparison) * 100:.1f}%)\n")
    f.write(f"  Casos donde CER mejoró: {(cer_diff > 0).sum()} ({(cer_diff > 0).sum() / len(df_comparison) * 100:.1f}%)\n")

print(f"   ✅ Estadísticas guardadas: {OUTPUT_STATS_TXT}")

# Resumen final
print("\n" + "=" * 60)
print("RESUMEN FINAL")
print("=" * 60)
print(f"\nTotal de muestras analizadas: {len(df_comparison)}")
print(f"\nMétricas Originales:")
print(f"  WER promedio: {df_comparison['wer'].mean():.2f}%")
print(f"  CER promedio: {df_comparison['cer'].mean():.2f}%")
print(f"\nMétricas Mejoradas:")
print(f"  WER promedio: {df_comparison['wer_improved'].mean():.2f}%")
print(f"  CER promedio: {df_comparison['cer_improved'].mean():.2f}%")
print(f"\nMejora Promedio:")
print(f"  WER: {wer_diff.mean():.2f} puntos porcentuales")
print(f"  CER: {cer_diff.mean():.2f} puntos porcentuales")
print(f"\n✅ Archivos generados:")
print(f"  - {OUTPUT_IMPROVED_CSV}")
print(f"  - {OUTPUT_STATS_TXT}")
print(f"  - {OUTPUT_PLOT_METRICS}")
print(f"  - {OUTPUT_PLOT_BOXPLOT}")
print("\n" + "=" * 60)


[8/8] Guardando resultados...
   ✅ CSV guardado: output\test_results\transcription_results_improved.csv
   ✅ Estadísticas guardadas: output\test_results\transcription_stats_improved.txt

RESUMEN FINAL

Total de muestras analizadas: 6040

Métricas Originales:
  WER promedio: 17.06%
  CER promedio: 7.08%

Métricas Mejoradas:
  WER promedio: 12.98%
  CER promedio: 6.18%

Mejora Promedio:
  WER: 4.08 puntos porcentuales
  CER: 0.91 puntos porcentuales

✅ Archivos generados:
  - output\test_results\transcription_results_improved.csv
  - output\test_results\transcription_stats_improved.txt
  - output\test_results\comparison_metrics.png
  - output\test_results\comparison_boxplot.png



In [16]:
# Análisis de casos con WER alto (> 70%)
print("=" * 60)
print("ANÁLISIS DE CASOS CON WER ALTO (> 70%)")
print("=" * 60)

# Cargar el CSV mejorado
df_improved = pd.read_csv("output/test_results/transcription_results_improved.csv")

# Filtrar casos con WER > 70%
df_high_wer = df_improved[df_improved['wer_improved'] > 70].copy()

print(f"\nTotal de casos con WER > 70%: {len(df_high_wer)}")
print(f"Porcentaje del total: {len(df_high_wer) / len(df_improved) * 100:.2f}%")

if len(df_high_wer) > 0:
    print(f"\nEstadísticas de WER en casos problemáticos:")
    print(df_high_wer['wer_improved'].describe())
    
    print(f"\nEstadísticas de CER en casos problemáticos:")
    print(df_high_wer['cer_improved'].describe())
    
    # Mostrar todos los casos con WER alto (ordenados por WER descendente)
    print("\n" + "=" * 60)
    print("TODOS LOS CASOS CON WER > 70% (ordenados por WER descendente)")
    print("=" * 60)
    
    df_high_wer_sorted = df_high_wer.sort_values('wer_improved', ascending=False)
    
    for idx, row in df_high_wer_sorted.iterrows():
        print(f"\n[{idx}] Audio: {row['audio_file']}")
        print(f"  Reference: {row['reference']}")
        print(f"  Transcribed: {row['transcribed']}")
        print(f"  WER: {row['wer_improved']:.2f}%, CER: {row['cer_improved']:.2f}%")
        print(f"  Substitutions: {row['substitutions_improved']}, Insertions: {row['insertions_improved']}, Deletions: {row['deletions_improved']}")
    
    # Guardar lista de archivos problemáticos
    problematic_files = df_high_wer_sorted[['audio_file', 'wer_improved', 'cer_improved', 'reference', 'transcribed']].copy()
    problematic_files.to_csv('output/test_results/high_wer_cases.csv', index=False, encoding='utf-8')
    print(f"\n✅ Lista de casos problemáticos guardada en: output/test_results/high_wer_cases.csv")
else:
    print("\n✅ No se encontraron casos con WER > 70%")

ANÁLISIS DE CASOS CON WER ALTO (> 70%)

Total de casos con WER > 70%: 226
Porcentaje del total: 3.69%

Estadísticas de WER en casos problemáticos:
count    226.000000
mean     106.726896
std       36.335743
min       71.428571
25%       92.857143
50%      100.000000
75%      100.000000
max      350.000000
Name: wer_improved, dtype: float64

Estadísticas de CER en casos problemáticos:
count    226.000000
mean      71.004965
std       36.447684
min        4.166667
25%       47.058824
50%       78.416149
75%       86.475034
max      266.666667
Name: cer_improved, dtype: float64

TODOS LOS CASOS CON WER > 70% (ordenados por WER descendente)

[1362] Audio: common_voice_es_42879647.mp3
  Reference: Fue catalogada para todos los públicos.
  Transcribed: hola a todos, estoy caminando por morroco pero estoy viviendo en españa y quiero tratar de traducir español, español, inglés, árabe
  WER: 350.00%, CER: 266.67%
  Substitutions: 6.0, Insertions: 15.0, Deletions: 0.0

[275] Audio: common_voice_

In [17]:
# Añadir columna para marcar archivos a eliminar
df_high_wer = pd.read_csv("output/test_results/high_wer_cases.csv")

# Añadir columna 'to_delete' (inicialmente False/0)
if 'to_delete' not in df_high_wer.columns:
    df_high_wer['to_delete'] = False
    df_high_wer.to_csv('output/test_results/high_wer_cases.csv', index=False, encoding='utf-8')
    print("✅ Columna 'to_delete' añadida al CSV")
    print("   Puedes editar el CSV y marcar con True/1 los archivos que quieres eliminar")
else:
    print("✅ La columna 'to_delete' ya existe")

✅ Columna 'to_delete' añadida al CSV
   Puedes editar el CSV y marcar con True/1 los archivos que quieres eliminar


In [18]:
# Script para eliminar archivos del dataset
import pandas as pd
import shutil
from pathlib import Path

# 1. Cargar lista de archivos marcados para eliminar
df_high_wer = pd.read_csv("output/test_results/high_wer_cases.csv")

# Convertir columna to_delete a booleano
df_high_wer['to_delete'] = df_high_wer['to_delete'].astype(str).str.strip()
df_high_wer['to_delete'] = df_high_wer['to_delete'].isin(['1', 'True', 'true', 'TRUE', 'yes', 'Yes', 'YES'])

files_to_delete = df_high_wer[df_high_wer['to_delete'] == True]['audio_file'].tolist()

if len(files_to_delete) == 0:
    print("⚠️  No se encontraron archivos marcados para eliminar")
    print("   Verifica que la columna 'to_delete' tenga valores 1 o True")
else:
    print(f"📋 Archivos a eliminar: {len(files_to_delete)}")
    
    # Rutas
    CLIPS_DIR = Path("data/cv-corpus-22.0-delta-2025-06-20/es/clips")
    TSV_FILE = Path("data/cv-corpus-22.0-delta-2025-06-20/es/other.tsv")
    
    deleted_clips = 0
    not_found_clips = []
    
    # 2. Eliminar archivos de audio
    print("\n[1/2] Eliminando archivos de audio...")
    for audio_file in files_to_delete:
        audio_path = CLIPS_DIR / audio_file
        if audio_path.exists():
            audio_path.unlink()
            deleted_clips += 1
        else:
            not_found_clips.append(audio_file)
    
    print(f"   ✅ Eliminados {deleted_clips} archivos de audio de la carpeta clips")
    if not_found_clips:
        print(f"   ⚠️  {len(not_found_clips)} archivos no encontrados en clips")
        if len(not_found_clips) <= 10:
            for f in not_found_clips:
                print(f"      - {f}")
        else:
            for f in not_found_clips[:10]:
                print(f"      - {f}")
            print(f"      ... y {len(not_found_clips) - 10} más")
    
    # 3. Eliminar entradas del TSV
    print("\n[2/2] Eliminando entradas del TSV...")
    if TSV_FILE.exists():
        df_tsv = pd.read_csv(TSV_FILE, delimiter='\t')
        original_count = len(df_tsv)
        df_tsv_filtered = df_tsv[~df_tsv['path'].isin(files_to_delete)]
        new_count = len(df_tsv_filtered)
        deleted_from_tsv = original_count - new_count
        
        if deleted_from_tsv > 0:
            df_tsv_filtered.to_csv(TSV_FILE, sep='\t', index=False, encoding='utf-8')
            print(f"   ✅ Eliminadas {deleted_from_tsv} entradas del TSV")
            print(f"   Registros restantes: {new_count} (antes: {original_count})")
        else:
            print(f"   ⚠️  No se encontraron entradas en el TSV para eliminar")
    else:
        print(f"   ⚠️  Archivo TSV no encontrado: {TSV_FILE}")
    
    # 4. Guardar lista de archivos eliminados
    with open('output/test_results/files_deleted.txt', 'w', encoding='utf-8') as f:
        f.write(f"Archivos eliminados: {len(files_to_delete)}\n")
        f.write(f"Fecha: {pd.Timestamp.now()}\n\n")
        for file in files_to_delete:
            f.write(f"{file}\n")
    
    print(f"\n✅ Proceso completado")
    print(f"   - Archivos de audio eliminados: {deleted_clips}")
    print(f"   - Entradas del TSV eliminadas: {deleted_from_tsv if TSV_FILE.exists() else 0}")
    print(f"   - Lista guardada en: output/test_results/files_deleted.txt")

📋 Archivos a eliminar: 171

[1/2] Eliminando archivos de audio...
   ✅ Eliminados 171 archivos de audio de la carpeta clips

[2/2] Eliminando entradas del TSV...
   ✅ Eliminadas 171 entradas del TSV
   Registros restantes: 5955 (antes: 6126)

✅ Proceso completado
   - Archivos de audio eliminados: 171
   - Entradas del TSV eliminadas: 171
   - Lista guardada en: output/test_results/files_deleted.txt
